# 02B — FSP Semantic Fragment Retrieval Validation — FINAL READY V4

Varian **semantic fragment retrieval** untuk analisis alternatif FSP.

Perbaikan V4:
- source Duri identik train–validation tidak langsung dianggap fatal;
- exact pair Duri–Indonesia lintas split tetap fatal;
- train row dengan full source identik terhadap query validation dikeluarkan dari retrieval;
- prompt dan format contoh identik dengan 02A;
- cosine search dihitung langsung dengan NumPy (tanpa FAISS) agar dependency lebih sederhana;
- similarity/fragment metadata hanya disimpan sebagai audit, tidak diberikan ke LLM;
- `k=2/4/6` berarti k contoh per query fragment;
- test tidak dibaca untuk pemilihan konfigurasi;
- fingerprint dataset disimpan;
- terminator Llama ditangani;
- chrF++ utama dan BLEU tie-breaker.

Jika semantic akhirnya dipilih, laporan harus menyebutnya sebagai **varian FSP dengan semantic fragment retrieval**, bukan reproduksi literal exact-FSP.


## 1. Instalasi

In [1]:
# Jalankan sekali pada instance Vast.ai baru.
# PyTorch/CUDA dari image Vast.ai TIDAK diinstal ulang.
%pip -q install -U --upgrade-strategy only-if-needed \
    "transformers==4.48.1" \
    "accelerate==1.2.1" \
    "huggingface_hub==0.27.1" \
    "sentence-transformers==3.4.1" \
    "sacrebleu==2.5.1" \
    "sentencepiece==0.2.0" \
    "protobuf>=4.25,<6" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "psutil>=7,<8" \
    "openpyxl>=3.1,<4" \
    "tqdm>=4.66,<5"

print("Dependensi 02B selesai.")
print("Jika package berubah, restart kernel SATU KALI lalu mulai lagi dari Environment.")


Note: you may need to restart the kernel to use updated packages.
Dependensi 02B selesai.
Jika package berubah, restart kernel SATU KALI lalu mulai lagi dari Environment.


## 2. Environment + audit dataset

In [2]:
from __future__ import annotations

import os, re, json, time, random, hashlib, gc, shutil
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from IPython.display import display
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sacrebleu.metrics import BLEU, CHRF

# ============================================================
# LOCK PENELITIAN
# ============================================================
PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

# HANYA path dataset final saat ini.
DATASET_DIR = PROJECT_ROOT / "Dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
VALID_PATH = DATASET_DIR / "validation.csv"
TEST_PATH = DATASET_DIR / "test.csv"   # hanya path; isi TEST TIDAK DIBACA.

CONFIG_DIR = PROJECT_ROOT / "Config"
ARTIFACT_DIR = PROJECT_ROOT / "Artifacts"
HASIL_DIR = PROJECT_ROOT / "Hasil"

for d in [CONFIG_DIR, ARTIFACT_DIR, HASIL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
EMBEDDING_MODEL_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

EXPECTED_TRAIN = 417
EXPECTED_VALID = 53
EXPECTED_TRAIN_CONV = 44
EXPECTED_VALID_CONV = 3

SEED = 42
DEVICE_INDEX = 0
DTYPE = torch.bfloat16

os.environ.setdefault("HF_HOME", "/workspace/.hf_home")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

HF_TOKEN = get_token() or os.getenv("HF_TOKEN","").strip() or None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}")

for p in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File dataset final tidak ditemukan: {p}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia.")
torch.cuda.set_device(DEVICE_INDEX)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU tidak mendukung BF16.")

if not HF_TOKEN:
    raise RuntimeError("Hugging Face belum login. Jalankan `hf auth login` di terminal.")

REQUIRED = ["row_id","conversation_no","turn_order","duri_text","indonesia_text"]

def load_tuning_split(path: Path, expected_rows: int) -> pd.DataFrame:
    df = pd.read_csv(path,dtype=str,keep_default_na=False,encoding="utf-8-sig")
    df.columns=[str(c).strip() for c in df.columns]
    missing=[c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: kolom wajib hilang {missing}")
    df=df[REQUIRED].copy()
    for c in REQUIRED:
        df[c]=df[c].astype(str).str.strip()
    if len(df)!=expected_rows:
        raise ValueError(f"{path.name}: expected {expected_rows}, found {len(df)}")
    if df["row_id"].eq("").any() or df["row_id"].duplicated().any():
        raise ValueError(f"{path.name}: row_id kosong/duplikat.")
    if df["duri_text"].eq("").any() or df["indonesia_text"].eq("").any():
        raise ValueError(f"{path.name}: terdapat teks kosong.")
    return df.reset_index(drop=True)

train_df=load_tuning_split(TRAIN_PATH,EXPECTED_TRAIN)
valid_df=load_tuning_split(VALID_PATH,EXPECTED_VALID)

if train_df["conversation_no"].nunique()!=EXPECTED_TRAIN_CONV:
    raise ValueError(
        f"Train conversation expected {EXPECTED_TRAIN_CONV}, "
        f"found {train_df['conversation_no'].nunique()}"
    )
if valid_df["conversation_no"].nunique()!=EXPECTED_VALID_CONV:
    raise ValueError(
        f"Validation conversation expected {EXPECTED_VALID_CONV}, "
        f"found {valid_df['conversation_no'].nunique()}"
    )

if set(train_df["conversation_no"]) & set(valid_df["conversation_no"]):
    raise RuntimeError("Conversation leakage train-validation.")

def norm_text(x: str) -> str:
    return re.sub(r"\s+"," ",str(x).strip().casefold())


# ============================================================
# TRAIN–VALIDATION OVERLAP AUDIT
# ============================================================
# Source Duri identik TIDAK otomatis fatal karena tuturan yang sama
# dapat muncul pada conversation berbeda.
#
# Yang fatal adalah pasangan Duri–Indonesia identik lintas split.
#
# Untuk FSP, train row dengan FULL SOURCE identik terhadap query
# validation akan dikeluarkan dari retrieval sehingga tidak terjadi
# shortcut berupa kalimat input lengkap yang sama.
# ============================================================
train_audit = train_df.copy()
valid_audit = valid_df.copy()

train_audit["_src_norm"] = train_audit["duri_text"].map(norm_text)
train_audit["_tgt_norm"] = train_audit["indonesia_text"].map(norm_text)

valid_audit["_src_norm"] = valid_audit["duri_text"].map(norm_text)
valid_audit["_tgt_norm"] = valid_audit["indonesia_text"].map(norm_text)

train_sources = set(train_audit["_src_norm"])
valid_sources = set(valid_audit["_src_norm"])
source_overlap = sorted(train_sources & valid_sources)

print("SOURCE OVERLAP TRAIN–VALIDATION :", len(source_overlap))

source_overlap_detail = pd.DataFrame()

if source_overlap:
    source_overlap_detail = (
        train_audit[
            train_audit["_src_norm"].isin(source_overlap)
        ][
            [
                "row_id",
                "conversation_no",
                "turn_order",
                "duri_text",
                "indonesia_text",
                "_src_norm",
                "_tgt_norm",
            ]
        ]
        .merge(
            valid_audit[
                valid_audit["_src_norm"].isin(source_overlap)
            ][
                [
                    "row_id",
                    "conversation_no",
                    "turn_order",
                    "duri_text",
                    "indonesia_text",
                    "_src_norm",
                    "_tgt_norm",
                ]
            ],
            on="_src_norm",
            how="inner",
            suffixes=("_train", "_valid"),
        )
    )

    source_overlap_detail["target_identik"] = (
        source_overlap_detail["_tgt_norm_train"]
        == source_overlap_detail["_tgt_norm_valid"]
    )

    print(
        "WARNING: ditemukan source Duri identik lintas split. "
        "Ini belum otomatis berarti duplicate pair."
    )
    display(
        source_overlap_detail[
            [
                "_src_norm",
                "row_id_train",
                "conversation_no_train",
                "duri_text_train",
                "indonesia_text_train",
                "row_id_valid",
                "conversation_no_valid",
                "duri_text_valid",
                "indonesia_text_valid",
                "target_identik",
            ]
        ]
    )
else:
    print("OK: tidak ada source identik train–validation.")

train_pairs = set(
    zip(
        train_audit["_src_norm"],
        train_audit["_tgt_norm"],
    )
)
valid_pairs = set(
    zip(
        valid_audit["_src_norm"],
        valid_audit["_tgt_norm"],
    )
)
exact_pair_overlap = sorted(train_pairs & valid_pairs)

print("EXACT PAIR OVERLAP TRAIN–VALIDATION :", len(exact_pair_overlap))

if exact_pair_overlap:
    if not source_overlap_detail.empty:
        exact_overlap_detail = source_overlap_detail[
            source_overlap_detail["target_identik"]
        ].copy()

        if not exact_overlap_detail.empty:
            display(
                exact_overlap_detail[
                    [
                        "_src_norm",
                        "row_id_train",
                        "conversation_no_train",
                        "indonesia_text_train",
                        "row_id_valid",
                        "conversation_no_valid",
                        "indonesia_text_valid",
                    ]
                ]
            )

    raise RuntimeError(
        "FATAL: terdapat pasangan Duri–Indonesia identik antara train "
        "dan validation. Split dataset harus diperbaiki sebelum eksperimen."
    )

print("PASS: tidak ada pasangan Duri–Indonesia identik train–validation.")

if source_overlap and not exact_pair_overlap:
    print(
        "CATATAN: source identik ada tetapi target tidak identik. "
        "Eksperimen dapat dilanjutkan. Retrieval FSP akan otomatis "
        "mengecualikan train row dengan full source identik."
    )

PROMPT_TEMPLATE_VERSION = "fsp_duri_indonesia_shared_prompt_v4"

def dataset_fingerprint() -> str:
    h=hashlib.sha256()
    for split_name,df in [("train",train_df),("validation",valid_df)]:
        ordered=df.sort_values("row_id",kind="stable")
        for r in ordered.itertuples(index=False):
            payload={
                "split":split_name,
                "row_id":r.row_id,
                "conversation_no":r.conversation_no,
                "turn_order":r.turn_order,
                "duri_text":r.duri_text,
                "indonesia_text":r.indonesia_text,
            }
            h.update(
                (json.dumps(payload,ensure_ascii=False,sort_keys=True,separators=(",",":"))+"\n")
                .encode("utf-8")
            )
    return h.hexdigest()

DATASET_FINGERPRINT=dataset_fingerprint()

def terminator_ids(tokenizer) -> List[int]:
    ids=[]
    if tokenizer.eos_token_id is not None:
        if isinstance(tokenizer.eos_token_id,(list,tuple)):
            ids.extend(int(x) for x in tokenizer.eos_token_id)
        else:
            ids.append(int(tokenizer.eos_token_id))
    for tok in ["<|eot_id|>","<|end_of_text|>"]:
        tid=tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid,int) and tid>=0 and tid!=tokenizer.unk_token_id and tid not in ids:
            ids.append(tid)
    return ids

def clean_translation(text: str) -> str:
    x=re.sub(r"\s+"," ",str(text or "").strip())
    prefixes=["Bahasa Indonesia:","Indonesia:","Terjemahan:","Hasil terjemahan:","Jawaban:"]
    changed=True
    while changed:
        changed=False
        for p in prefixes:
            if x.casefold().startswith(p.casefold()):
                x=x[len(p):].strip()
                changed=True
    return x

BASE_INSTRUCTION=(
    "Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia. "
    "Gunakan contoh pasangan terjemahan yang diberikan sebagai konteks. "
    "Keluarkan hanya terjemahan bahasa Indonesia tanpa penjelasan tambahan."
)

print("PROJECT_ROOT        :",PROJECT_ROOT)
print("TRAIN               :",len(train_df),"rows /",train_df["conversation_no"].nunique(),"conversation")
print("VALIDATION          :",len(valid_df),"rows /",valid_df["conversation_no"].nunique(),"conversation")
print("TEST                : TIDAK DIBACA")
print("DATASET FINGERPRINT :",DATASET_FINGERPRINT)
print("GPU                 :",torch.cuda.get_device_name(DEVICE_INDEX))
print("MODEL               :",MODEL_ID)


SOURCE OVERLAP TRAIN–VALIDATION : 1


,_src_norm,row_id_train,conversation_no_train,duri_text_train,indonesia_text_train,row_id_valid,conversation_no_valid,duri_text_valid,indonesia_text_valid,target_identik
0,waqdingraka ke undinaq?,C01_T05_A3,1,Waqdingraka ke undinaq?,Bolehkah jika saya ikut?,C40_T03_A2,40,Waqdingraka ke undinaq?,Bolehkah saya ikut?,False


EXACT PAIR OVERLAP TRAIN–VALIDATION : 0
PASS: tidak ada pasangan Duri–Indonesia identik train–validation.
CATATAN: source identik ada tetapi target tidak identik. Eksperimen dapat dilanjutkan. Retrieval FSP akan otomatis mengecualikan train row dengan full source identik.
PROJECT_ROOT        : /workspace/10122233_CikitaMeydilvira_Skripsi
TRAIN               : 417 rows / 44 conversation
VALIDATION          : 53 rows / 3 conversation
TEST                : TIDAK DIBACA
DATASET FINGERPRINT : 6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc
GPU                 : NVIDIA GeForce RTX 4090
MODEL               : meta-llama/Llama-3.1-8B-Instruct


## 3. Train fragment bank + FAISS

In [3]:
from sentence_transformers import SentenceTransformer

MAX_FRAGMENT_LEN=7
K_VALUES=[2,4,6]
SEARCH_K=256

def contiguous_spans(text: str) -> List[Tuple[str,int,int]]:
    tt=norm_text(text).split()
    out=[]
    for n in range(1,min(MAX_FRAGMENT_LEN,len(tt))+1):
        for s in range(len(tt)-n+1):
            out.append((" ".join(tt[s:s+n]),n,s))
    return out

records=[]
for r in tqdm(train_df.itertuples(index=False),total=len(train_df),desc="Build train fragment bank"):
    seen=set()
    for frag,n,s in contiguous_spans(r.duri_text):
        key=(frag,n,s)
        if key in seen:
            continue
        seen.add(key)
        records.append({
            "row_id":r.row_id,
            "fragment":frag,
            "length":n,
            "start":s,
            "duri_text":r.duri_text,
            "indonesia_text":r.indonesia_text,
            "full_source_norm":norm_text(r.duri_text),
        })
frag_df=pd.DataFrame(records)

embedder=SentenceTransformer(EMBEDDING_MODEL_ID,device="cuda")
frag_emb=embedder.encode(
    frag_df["fragment"].tolist(),
    batch_size=128,normalize_embeddings=True,
    convert_to_numpy=True,show_progress_bar=True
).astype("float32")

if frag_emb.ndim != 2 or len(frag_emb) != len(frag_df):
    raise RuntimeError(
        f"Shape embedding tidak sesuai: {frag_emb.shape}, rows={len(frag_df)}"
    )

sem_art=ARTIFACT_DIR/"fsp_semantic"
sem_art.mkdir(parents=True,exist_ok=True)
frag_df.to_csv(sem_art/"train_fragments.csv",index=False,encoding="utf-8-sig")
np.save(sem_art/"train_fragment_embeddings.npy",frag_emb)
print("Fragments:",len(frag_df))
print("Embedding:",frag_emb.shape)


Build train fragment bank: 100%|██████████| 417/417 [00:00<00:00, 5415.24it/s]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Fragments: 11393
Embedding: (11393, 768)


## 4. Semantic retrieval per non-overlap query fragment

In [4]:
def query_fragments_nonoverlap(text: str) -> List[Tuple[str,int,int]]:
    # Longest-first segmentation, maksimal 7 token.
    tt=norm_text(text).split()
    occupied=set()
    chosen=[]
    for n in range(min(MAX_FRAGMENT_LEN,len(tt)),0,-1):
        for s in range(len(tt)-n+1):
            pos=set(range(s,s+n))
            if pos & occupied:
                continue
            chosen.append((" ".join(tt[s:s+n]),n,s))
            occupied |= pos
    return chosen

def semantic_retrieve(source: str,k_per_fragment: int) -> Dict[str,Any]:
    query_norm=norm_text(source)
    qfrags=query_fragments_nonoverlap(source)
    if not qfrags:
        return {"fragments":[],"examples":[]}

    qemb=embedder.encode(
        [x[0] for x in qfrags],
        batch_size=32,normalize_embeddings=True,
        convert_to_numpy=True,show_progress_bar=False
    ).astype("float32")

    # Karena seluruh embedding sudah L2-normalized,
    # dot product = cosine similarity.
    scores=qemb @ frag_emb.T

    seen_rows=set()
    examples=[]
    fragment_meta=[]

    for qi,(qfrag,qlen,qstart) in enumerate(qfrags):
        candidates={}

        sim=scores[qi]
        sk=min(SEARCH_K,len(sim))

        if sk < len(sim):
            top_ids=np.argpartition(-sim,sk-1)[:sk]
            top_ids=top_ids[np.argsort(-sim[top_ids],kind="stable")]
        else:
            top_ids=np.argsort(-sim,kind="stable")

        for fi in top_ids:
            fi=int(fi)
            tr=frag_df.iloc[fi]

            # Anti-shortcut:
            # FULL SOURCE train identik dengan query validation
            # tidak boleh masuk contoh semantic FSP.
            if str(tr["full_source_norm"]) == query_norm:
                continue

            rid=str(tr["row_id"])
            score=float(sim[fi])
            # Satu train row hanya menyumbang best fragment untuk query fragment ini.
            if rid not in candidates or score>candidates[rid]["score"]:
                candidates[rid]={
                    "row_id":rid,
                    "score":score,
                    "query_fragment":qfrag,
                    "matched_train_fragment":tr["fragment"],
                    "duri_text":tr["duri_text"],
                    "indonesia_text":tr["indonesia_text"],
                }

        ranked=sorted(
            candidates.values(),
            key=lambda x:(-x["score"],x["row_id"])
        )
        added=0
        selected_scores=[]
        for c in ranked:
            if c["row_id"] in seen_rows:
                continue
            examples.append(c)
            seen_rows.add(c["row_id"])
            selected_scores.append(c["score"])
            added+=1
            if added>=k_per_fragment:
                break

        fragment_meta.append({
            "query_fragment":qfrag,
            "start":qstart,
            "length":qlen,
            "retrieved_count":added,
            "mean_selected_similarity":(
                float(np.mean(selected_scores)) if selected_scores else np.nan
            ),
            "top_selected_similarity":(
                float(np.max(selected_scores)) if selected_scores else np.nan
            ),
        })

    return {"fragments":fragment_meta,"examples":examples}

preview=semantic_retrieve(valid_df.iloc[0]["duri_text"],2)
print("Query:",valid_df.iloc[0]["duri_text"])
display(pd.DataFrame(preview["fragments"]))
display(pd.DataFrame(preview["examples"]).head(12))


Query: Indara sangantaq?


,query_fragment,start,length,retrieved_count,mean_selected_similarity,top_selected_similarity
0,indara sangantaq?,0,2,2,0.796667,0.805305


,row_id,score,query_fragment,matched_train_fragment,duri_text,indonesia_text
0,C42_T03_A2,0.805305,indara sangantaq?,indara nasolan,Indara nasolan mamana Rusna tonna keanak?,Siapakah yang menemani mamanya Rusna waktu mel...
1,C23_T09_A5,0.788030,indara sangantaq?,indara mangbuah jio baraqbah?,Indara mangbuah jio baraqbah?,Siapakah yang mengolah tanah di kebun?


## 5. Prompt + model + validation

In [5]:
def build_prompt(source: str,z: Dict[str,Any]) -> str:
    blocks=[BASE_INSTRUCTION,""]
    if z["examples"]:
        blocks.append("Contoh:")
        for i,e in enumerate(z["examples"],1):
            # IDENTIK dengan 02A: score/fragment TIDAK masuk prompt.
            blocks += [
                f"[Contoh {i}]",
                f"Duri: {e['duri_text']}",
                f"Indonesia: {e['indonesia_text']}",
                ""
            ]
    blocks += ["Teks yang harus diterjemahkan:",f"Duri: {source}","Indonesia:"]
    return "\n".join(blocks).strip()

# Pindahkan embedder ke CPU agar VRAM diprioritaskan untuk Llama.
embedder.to("cpu")
gc.collect()
torch.cuda.empty_cache()

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,token=HF_TOKEN,use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id=tokenizer.eos_token_id

model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,token=HF_TOKEN,torch_dtype=DTYPE,
    low_cpu_mem_usage=True,device_map={"":DEVICE_INDEX}
)
model.eval()
model.config.use_cache=True
if getattr(model,"generation_config",None) is not None:
    model.generation_config.do_sample=False
    model.generation_config.temperature=None
    model.generation_config.top_p=None
    model.generation_config.top_k=None

EOS_IDS=terminator_ids(tokenizer)

def generate_semantic(source: str,k: int) -> Dict[str,Any]:
    z=semantic_retrieve(source,k)
    prompt=build_prompt(source,z)
    enc=tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}],
        tokenize=True,add_generation_prompt=True,
        return_tensors="pt",return_dict=True
    )
    enc={a:b.to(f"cuda:{DEVICE_INDEX}") for a,b in enc.items()}
    plen=int(enc["input_ids"].shape[-1])
    with torch.inference_mode():
        y=model.generate(
            **enc,max_new_tokens=128,do_sample=False,num_beams=1,
            repetition_penalty=1.0,use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=EOS_IDS if EOS_IDS else None,
        )
    raw=tokenizer.decode(
        y[0,plen:],skip_special_tokens=True,clean_up_tokenization_spaces=False
    )
    sims=[float(e["score"]) for e in z["examples"]]
    return {
        "output":clean_translation(raw),
        "fragment_count":len(z["fragments"]),
        "example_count":len(z["examples"]),
        "mean_similarity":float(np.mean(sims)) if sims else np.nan,
        "top_similarity":float(np.max(sims)) if sims else np.nan,
        "prompt_tokens":plen,
        "fragments_json":json.dumps(z["fragments"],ensure_ascii=False),
        "examples_json":json.dumps(z["examples"],ensure_ascii=False),
    }

RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR=HASIL_DIR/"fsp_semantic_validation"/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=False)

bleu_metric=BLEU(tokenize="13a")
chrf_metric=CHRF(word_order=2)

all_frames=[]
summary_rows=[]

for k in K_VALUES:
    rows=[]
    for r in tqdm(valid_df.itertuples(index=False),total=len(valid_df),desc=f"Semantic k={k}"):
        try:
            z=generate_semantic(r.duri_text,k)
            rec={
                "row_id":r.row_id,"k_per_fragment":k,
                "source":r.duri_text,"reference":r.indonesia_text,
                **z,"error":""
            }
        except Exception as e:
            rec={
                "row_id":r.row_id,"k_per_fragment":k,
                "source":r.duri_text,"reference":r.indonesia_text,
                "output":"","fragment_count":0,"example_count":0,
                "mean_similarity":np.nan,"top_similarity":np.nan,
                "prompt_tokens":np.nan,"fragments_json":"[]","examples_json":"[]",
                "error":f"{type(e).__name__}: {e}"
            }
        rows.append(rec)
        pd.DataFrame(rows).to_csv(
            RUN_DIR/f"predictions_k{k}.csv",index=False,encoding="utf-8-sig"
        )

    g=pd.DataFrame(rows)
    all_frames.append(g)
    ok=g["output"].astype(str).str.strip().ne("")
    complete=(len(g)==EXPECTED_VALID and int(ok.sum())==EXPECTED_VALID)
    bleu=chrf=np.nan
    if complete:
        hyps=g["output"].tolist()
        refs=g["reference"].tolist()
        bleu=float(bleu_metric.corpus_score(hyps,[refs]).score)
        chrf=float(chrf_metric.corpus_score(hyps,[refs]).score)
    summary_rows.append({
        "k_per_fragment":k,
        "complete":complete,
        "success":int(ok.sum()),
        "chrfpp":chrf,
        "bleu":bleu,
        "mean_examples":float(g["example_count"].mean()),
        "mean_similarity":float(g["mean_similarity"].mean()),
        "mean_prompt_tokens":float(g["prompt_tokens"].mean()),
    })

summary=pd.DataFrame(summary_rows)
summary.to_csv(RUN_DIR/"summary.csv",index=False,encoding="utf-8-sig")
pd.concat(all_frames,ignore_index=True).to_csv(
    RUN_DIR/"predictions_all.csv",index=False,encoding="utf-8-sig"
)
display(summary.sort_values(
    ["complete","chrfpp","bleu","mean_prompt_tokens"],
    ascending=[False,False,False,True]
))

eligible=summary[summary["complete"]].copy()
if eligible.empty:
    raise RuntimeError("Tidak ada konfigurasi semantic yang lengkap 53/53.")

winner=eligible.sort_values(
    ["chrfpp","bleu","mean_prompt_tokens"],
    ascending=[False,False,True]
).iloc[0]

cfg={
    "method":"fsp_semantic_fragment_modified",
    "paper_status":"modified semantic fragment retrieval; not literal exact-FSP",
    "model_id":MODEL_ID,
    "embedding_model":EMBEDDING_MODEL_ID,
    "dataset_fingerprint":DATASET_FINGERPRINT,
    "prompt_template_version":PROMPT_TEMPLATE_VERSION,
    "source_overlap_count":len(source_overlap),
    "exact_pair_overlap_count":len(exact_pair_overlap),
    "identical_full_source_train_rows_excluded_from_retrieval":True,
    "train_rows":len(train_df),
    "validation_rows":len(valid_df),
    "test_read":False,
    "max_fragment_len":MAX_FRAGMENT_LEN,
    "tested_k_values":K_VALUES,
    "k_per_fragment":int(winner["k_per_fragment"]),
    "validation_chrfpp":float(winner["chrfpp"]),
    "validation_bleu":float(winner["bleu"]),
    "mean_prompt_tokens":float(winner["mean_prompt_tokens"]),
    "selection_rule":"complete -> chrF++ desc -> BLEU desc -> mean_prompt_tokens asc",
    "validation_run_dir":str(RUN_DIR),
}
(CONFIG_DIR/"selected_fsp_semantic_config.json").write_text(
    json.dumps(cfg,ensure_ascii=False,indent=2),encoding="utf-8"
)
print(json.dumps(cfg,ensure_ascii=False,indent=2))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 14.38 MiB is free. Process 3827615 has 16.36 GiB memory in use. Process 3833312 has 7.13 GiB memory in use. Of the allocated memory 6.67 GiB is allocated by PyTorch, and 7.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)